In [6]:
"""
Response-Based Knowledge Distillation — ResNet50 (teacher) → ResNet18 (student) / CIFAR-10
============================================================================================

WHAT THIS SCRIPT DOES
=====================
Implements three response-based KD variants, sweeping over all three:

  1. KD-KL  — Hinton 2015: KL divergence on temperature-scaled soft probabilities
                 L = (1-α)·CE(y, p_S) + α·τ²·KL(p_T ‖ p_S)

  2. KD-MSE — Logit MSE: mean-squared error directly on raw logits (no temperature)
                 L = (1-α)·CE(y, p_S) + α·MSE(z_T, z_S)

  3. DIST   — Pearson-correlation: matches rank order of predictions, not values
                 L = (1-α)·CE(y, p_S) + α·(1 - pearson_corr(z_T, z_S))

ARCHITECTURE
============
  Teacher : ResNet50 — same modified variant as the CIFAR-10 baseline
              conv1 → 3×3 stride-1, maxpool → Identity, fc → 10 classes
              Loaded from: ../__2__baseline_resnet50_cifar10.pth  (FROZEN)

  Student : ResNet18 with identical CIFAR-10 head modifications
              conv1 → 3×3 stride-1, maxpool → Identity, fc → 10 classes
              Trained from scratch

OUTPUT
======
  __1__KD_response_based.json  — mirrors QAT JSON structure for direct comparison
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch  : {torch.__version__}", flush=True)
print(f"[init] Device   : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU      : {torch.cuda.get_device_name(0)}", flush=True)
    cap = torch.cuda.get_device_capability()
    print(f"[init] Compute  : SM{cap[0]}{cap[1]}", flush=True)

# ── Config ────────────────────────────────────────────────────────────────────
TEACHER_PATH          = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__1__KD_response_based.json"

BATCH_SIZE   = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS  = 0
PIN_MEMORY   = DEVICE.type == "cuda"
NUM_CLASSES  = 10
EPOCHS       = 10        # enough for student to converge with KD guidance
LR           = 0.1
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
USE_AMP      = DEVICE.type == "cuda"

# KD hyperparameters — well-validated defaults from literature
TEMPERATURE  = 4.0   # τ: softens teacher distribution to reveal dark knowledge
ALPHA        = 0.9   # weight on the KD loss term  (1-α on hard CE loss)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] Epochs   : {EPOCHS}  Batch: {BATCH_SIZE}", flush=True)
print(f"[init] τ={TEMPERATURE}  α={ALPHA}  LR={LR}  AMP={USE_AMP}", flush=True)

[init] PyTorch  : 2.11.0+cu128
[init] Device   : cuda
[init] GPU      : NVIDIA GeForce RTX 5070 Laptop GPU
[init] Compute  : SM120
[init] Epochs   : 10  Batch: 128
[init] τ=4.0  α=0.9  LR=0.1  AMP=True


In [7]:
# ── Model builders ────────────────────────────────────────────────────────────
def build_resnet50(num_classes: int = 10) -> nn.Module:
    """Exact same architecture used for the CIFAR-10 baseline."""
    m = models.resnet50(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def build_resnet18(num_classes: int = 10) -> nn.Module:
    """Student — ResNet18 with identical CIFAR-10 modifications."""
    m = models.resnet18(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m

def load_teacher(path: str) -> nn.Module:
    print(f"[teacher] Loading from {path} ...", flush=True)
    teacher = build_resnet50(NUM_CLASSES)
    teacher.load_state_dict(torch.load(path, map_location="cpu"))
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad_(False)   # hard freeze — teacher never updates
    teacher = teacher.to(DEVICE)
    print("[teacher] Loaded and frozen.", flush=True)
    return teacher

# ── Data ──────────────────────────────────────────────────────────────────────
def get_loaders():
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    train_ds = torchvision.datasets.CIFAR10("../data", train=True,  download=True, transform=train_tf)
    test_ds  = torchvision.datasets.CIFAR10("../data", train=False, download=True, transform=test_tf)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_ld  = DataLoader(test_ds,  batch_size=512,        shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    print(f"[data] Train batches: {len(train_ld)}  Test batches: {len(test_ld)}", flush=True)
    return train_ld, test_ld

In [8]:
# ── KD Loss functions ─────────────────────────────────────────────────────────
def loss_kd_kl(logits_s, logits_t, labels, tau=TEMPERATURE, alpha=ALPHA):
    """
    Hinton 2015 KD loss — temperature-scaled KL divergence.
    L = (1-α)·CE(y, p_S) + α·τ²·KL(p_T ‖ p_S)

    The τ² factor compensates for gradient magnitude reduction at high temperature:
    softmax outputs shrink by ~1/τ², so τ² restores the scale.
    """
    ce  = F.cross_entropy(logits_s, labels)
    p_t = F.softmax(logits_t / tau, dim=1)           # teacher soft distribution
    p_s = F.log_softmax(logits_s / tau, dim=1)       # student log-softmax
    kl  = F.kl_div(p_s, p_t, reduction="batchmean") * (tau ** 2)
    return (1.0 - alpha) * ce + alpha * kl

def loss_kd_mse(logits_s, logits_t, labels, alpha=ALPHA):
    """
    Logit-MSE KD (Hinton 2022 revisit).
    L = (1-α)·CE(y, p_S) + α·MSE(z_T, z_S)

    Works on raw unnormalized logits — no temperature needed.
    Preserves absolute logit magnitude; often outperforms KL-div variant.
    """
    ce  = F.cross_entropy(logits_s, labels)
    mse = F.mse_loss(logits_s, logits_t)
    return (1.0 - alpha) * ce + alpha * mse

def loss_dist(logits_s, logits_t, labels, alpha=ALPHA):
    """
    DIST — Pearson correlation loss (Huang 2022).
    L = (1-α)·CE(y, p_S) + α·(1 - mean_pearson(z_T, z_S))

    Matches rank ordering of class scores per sample rather than absolute values.
    More robust to teacher–student capacity gap and scale differences.
    """
    ce = F.cross_entropy(logits_s, labels)
    # Centre per-sample (subtract mean across class dimension)
    s = logits_s - logits_s.mean(dim=1, keepdim=True)
    t = logits_t - logits_t.mean(dim=1, keepdim=True)
    # Pearson = cosine similarity of centred vectors
    corr = (F.normalize(s, dim=1) * F.normalize(t, dim=1)).sum(dim=1).mean()
    return (1.0 - alpha) * ce + alpha * (1.0 - corr)

LOSS_FNS = {
    # "KD-KL" : loss_kd_kl,
    # "KD-MSE": loss_kd_mse,
    "DIST"  : loss_dist,
}

In [9]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device) -> dict:
    model.eval()
    preds, labels = [], []
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        preds.extend(model(x).argmax(1).cpu().numpy())
        labels.extend(y.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_ms(model: nn.Module, device, batch_size=128, runs=20) -> float:
    m = model.eval().to(device)
    dummy = torch.randn(batch_size, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(3): m(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000

def sync():
    if torch.cuda.is_available(): torch.cuda.synchronize()
    
    
def measure_inference_detailed(model: nn.Module, device) -> dict:
    model = model.eval().to(device)
    is_gpu = device.type == "cuda"

    # ── Single image ──────────────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)

    # ── Batch 128 ─────────────────────────────────────────────
    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }
    
def compute_flops(model: nn.Module) -> float:
    from thop import profile as thop_profile
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None

In [10]:
# ── Core training loop ────────────────────────────────────────────────────────
def run_kd(teacher, train_loader, test_loader,
           method_name: str, baseline_acc: float, teacher_size_mb: float) -> dict:
    print(f"\n  ┌─ [{method_name}]", flush=True)
    loss_fn = LOSS_FNS[method_name]

    student   = build_resnet18(NUM_CLASSES).to(DEVICE)
    optimizer = torch.optim.SGD(student.parameters(), lr=LR,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.amp.GradScaler(enabled=USE_AMP)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    epoch_train_acc = []
    t_start = time.perf_counter()

    for epoch in range(EPOCHS):
        student.train()
        correct = total = 0
        running_loss = 0.0
        t_ep = time.perf_counter()

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs  = inputs.to(DEVICE,  non_blocking=True)
            targets = targets.to(DEVICE, non_blocking=True)

            # Teacher inference — no gradients, frozen weights
            with torch.no_grad():
                logits_t = teacher(inputs)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits_s = student(inputs)
                loss     = loss_fn(logits_s, logits_t, targets)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            correct      += (logits_s.detach().argmax(1) == targets).sum().item()
            total        += targets.size(0)
            running_loss += loss.item()

            if (batch_idx + 1) % 100 == 0:
                elapsed = time.perf_counter() - t_ep
                eta     = elapsed / (batch_idx + 1) * (len(train_loader) - batch_idx - 1)
                print(f"      [{method_name}] ep {epoch+1}/{EPOCHS} "
                      f"b {batch_idx+1}/{len(train_loader)}  "
                      f"loss={running_loss/(batch_idx+1):.4f}  "
                      f"acc={correct/total:.4f}  ETA={eta:.0f}s", flush=True)

        scheduler.step()
        acc = correct / total
        epoch_train_acc.append(round(acc, 6))
        sync()
        ep_time = time.perf_counter() - t_ep
        print(f"      [{method_name}] epoch {epoch+1}/{EPOCHS} DONE  "
              f"acc={acc:.4f}  loss={running_loss/len(train_loader):.4f}  "
              f"time={ep_time:.1f}s  ETA_remaining={ep_time*(EPOCHS-epoch-1):.0f}s", flush=True)

    sync()
    train_time_s = time.perf_counter() - t_start
    peak_gpu_mb  = (torch.cuda.max_memory_allocated() / 1024**2
                    if DEVICE.type == "cuda" else None)

    print(f"      [{method_name}] Evaluating ...", flush=True)
    metrics  = evaluate(student, test_loader, DEVICE)
    inf      = measure_inference_detailed(student, DEVICE)
    size_mb  = model_size_mb(student)
    acc_drop = baseline_acc - metrics["accuracy"]
    flops_G  = compute_flops(student)
    total_params = param_count(student)
    
    # ── Save model ────────────────────────────────────────────
    save_dir = f"__1__saved_models_KD"
    os.makedirs(save_dir, exist_ok=True)
    save_path = f"{save_dir}/student_resnet18_{method_name}.pth"
    torch.save(student.state_dict(), save_path)
    print(f"      [{method_name}] ✓ Saved → {save_path}", flush=True)

    print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
          f"Size={size_mb:.2f}MB  Single={inf['single_img_gpu_ms']:.2f}ms  "
          f"Throughput={inf['throughput_imgs_sec']:.0f}imgs/s  "
          f"Train={train_time_s:.1f}s", flush=True)

    return {
        # identification
        "method"              : "response_based_kd",
        "variant"             : method_name,
        "teacher"             : "resnet50_cifar10_modified",
        "student"             : "resnet18_cifar10_modified",
        "dataset"             : "CIFAR-10",
        # hyperparameters
        "epochs"              : EPOCHS,
        "lr"                  : LR,
        "momentum"            : MOMENTUM,
        "weight_decay"        : WEIGHT_DECAY,
        "temperature"         : TEMPERATURE if method_name == "KD-KL" else "N/A",
        "alpha"               : ALPHA,
        "lr_schedule"         : "cosine",
        "train_device"        : str(DEVICE),
        "use_amp"             : USE_AMP,
        # training stats
        "train_time_s"        : round(train_time_s, 2),
        "epoch_train_acc"     : epoch_train_acc,
        "peak_gpu_memory_mb"  : round(peak_gpu_mb, 2) if peak_gpu_mb else None,
        # ── Standardised metrics block ────────────────────────
        "accuracy"            : round(metrics["accuracy"],  6),
        "precision"           : round(metrics["precision"], 6),
        "recall"              : round(metrics["recall"],    6),
        "f1"                  : round(metrics["f1"],        6),
        "accuracy_drop"       : round(acc_drop, 6),
        "params"              : total_params,
        "params_nonzero"      : total_params,   # KD = dense model, no zeroing
        "sparsity_actual"     : 0.0,
        "size_mb"             : round(size_mb, 4),
        "flops_G"             : flops_G,
        "inference_ms"        : inf,
        # size comparisons
        "teacher_size_mb"     : round(teacher_size_mb, 4),
        "compression_ratio"   : round(teacher_size_mb / size_mb, 4) if size_mb else None,
        # meta
        "saved_model_path"    : save_path,
        "description"         : (
            f"Response-based KD ({method_name}): ResNet50→ResNet18, CIFAR-10, "
            f"τ={TEMPERATURE}, α={ALPHA}, epochs={EPOCHS}, "
            f"device={DEVICE}, amp={USE_AMP}"
        ),
        "status": "ok",
    }


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*60}")
    print("  Response-Based KD — ResNet50 → ResNet18 / CIFAR-10")
    print(f"  Device : {DEVICE}  |  Epochs: {EPOCHS}  |  Batch: {BATCH_SIZE}")
    print(f"  τ={TEMPERATURE}  α={ALPHA}  AMP={USE_AMP}")
    print(f"{'='*60}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline accuracy : {baseline_acc:.4f}\n", flush=True)

    teacher         = load_teacher(TEACHER_PATH)
    teacher_size_mb = model_size_mb(teacher)
    print(f"  Teacher size      : {teacher_size_mb:.2f} MB", flush=True)

    train_loader, test_loader = get_loaders()

    results = []
    for method_name in LOSS_FNS:
        try:
            row = run_kd(teacher, train_loader, test_loader,
                         method_name, baseline_acc, teacher_size_mb)
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f"  └─ FAILED [{method_name}]: {e}", flush=True)
            row = {
                "method": "response_based_kd", "variant": method_name,
                "status": "failed", "error": str(e),
                "accuracy": None, "accuracy_drop": None,
                "size_disk_mb": None,
            }
        results.append(row)

    valid = [r for r in results if r.get("accuracy") is not None]
    best  = min(valid, key=lambda r: r["accuracy_drop"]) if valid else None

    report = {
        "experiment"      : "response_based_knowledge_distillation",
        "teacher"         : "resnet50_cifar10_modified",
        "student"         : "resnet18_cifar10_modified",
        "dataset"         : "CIFAR-10",
        "train_device"    : str(DEVICE),
        "use_amp"         : USE_AMP,
        "epochs"          : EPOCHS,
        "batch_size"      : BATCH_SIZE,
        "temperature"     : TEMPERATURE,
        "alpha"           : ALPHA,
        "variants_tested" : list(LOSS_FNS.keys()),
        "baseline"        : baseline,
        "teacher_size_mb" : round(teacher_size_mb, 4),
        "best_variant"    : best["variant"] if best else None,
        "best_config": {
            "variant"       : best["variant"]       if best else None,
            "accuracy"      : best["accuracy"]      if best else None,
            "accuracy_drop" : best["accuracy_drop"] if best else None,
            "size_mb"  : best["size_mb"]  if best else None,
            "inference_ms"  : best["inference_ms"]  if best else None,
        } if best else {},
        "results": results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    if best:
        print(f"  Best variant     : {best['variant']}")
        print(f"  Accuracy         : {best['accuracy']:.4f}  (drop={best['accuracy_drop']:+.4f})")
        print(f"  Student size     : {best['size_mb']:.2f} MB  "
              f"(teacher={teacher_size_mb:.2f} MB, "
              f"ratio={teacher_size_mb/best['size_mb']:.2f}×)")
        print(f"  Inference        : {best['inference_ms']:.1f} ms")


if __name__ == "__main__":
    main()


  Response-Based KD — ResNet50 → ResNet18 / CIFAR-10
  Device : cuda  |  Epochs: 10  |  Batch: 128
  τ=4.0  α=0.9  AMP=True

  Baseline accuracy : 0.9320

[teacher] Loading from ../__2__baseline_resnet50_cifar10.pth ...
[teacher] Loaded and frozen.
  Teacher size      : 90.03 MB
[data] Train batches: 391  Test batches: 20

  ┌─ [DIST]
      [DIST] ep 1/10 b 100/391  loss=0.8457  acc=0.2944  ETA=31s
      [DIST] ep 1/10 b 200/391  loss=0.7514  acc=0.3690  ETA=20s
      [DIST] ep 1/10 b 300/391  loss=0.6893  acc=0.4241  ETA=10s
      [DIST] epoch 1/10 DONE  acc=0.4612  loss=0.6453  time=41.5s  ETA_remaining=373s
      [DIST] ep 2/10 b 100/391  loss=0.4463  acc=0.6267  ETA=31s
      [DIST] ep 2/10 b 200/391  loss=0.4196  acc=0.6478  ETA=21s
      [DIST] ep 2/10 b 300/391  loss=0.4023  acc=0.6603  ETA=10s
      [DIST] epoch 2/10 DONE  acc=0.6735  loss=0.3863  time=42.0s  ETA_remaining=336s
      [DIST] ep 3/10 b 100/391  loss=0.2967  acc=0.7466  ETA=33s
      [DIST] ep 3/10 b 200/391  los

TypeError: unsupported format string passed to dict.__format__